<div class="title-slide">

# Make the Most of What You’ve Got: Deep Learning Optimization with PyTorch Profiler and Holistic Trace Analysis 

## Collin Wilson

</div>

# Getting started with this demo

### This talk is given in a Jupyter notebook format. You can download it and all the materials to recreate the demo on GitHub:

```
git clone https://github.com/cwilson51/GIS-PP-and-HTA.git
```



# What we'll cover today

- ## PyTorch Profiler revisited
    - ### Inline profiling
    - ### Generating traces
- ## Examining traces and simple optimizations
- ## Holistic trace analysis (HTA)
    - ### Single GPU
    - ### Distributed workloads


# Working example - CPU

## ResNet50 on random inputs

In [18]:
import torch
from torchvision.models import resnet50
from torchvision import transforms
from torch.profiler import profile, record_function, ProfilerActivity, schedule

model = resnet50()
inputs = torch.randn(1, 3, 224, 224)
labels = torch.randn(1, 1000)

loss_fn = torch.nn.CrossEntropyLoss()
optimizer = torch.optim.SGD(model.parameters(), lr=0.001, momentum=0.9)

with profile(activities=[ProfilerActivity.CPU, ProfilerActivity.GPU], record_shapes=True) as prof:

    model.train()
    with record_function("model_training"):
        outputs = model(inputs)
        loss = loss_fn(outputs, labels)
        loss.backward()
        optimizer.step()
        
    model.eval()
    with record_function("model_inference"):
        model(inputs)

USDT:2026-11-13 11:04:38 95876:14157365 SyncActivityProfilerHandler.cpp:39] profiler_start
USDT:2026-11-13 11:04:38 95876:14157365 SyncActivityProfilerHandler.cpp:46] profiler_stop


In [22]:
print(prof.key_averages().table(sort_by="cpu_time_total", row_limit=20))

-------------------------------------------------------  ------------  ------------  ------------  ------------  ------------  ------------  
                                                   Name    Self CPU %      Self CPU   CPU total %     CPU total  CPU time avg    # of Calls  
-------------------------------------------------------  ------------  ------------  ------------  ------------  ------------  ------------  
                                         model_training        33.61%      56.403ms        88.36%     148.263ms     148.263ms             1  
                                           aten::conv2d         0.76%       1.273ms        21.07%      35.363ms     333.615us           106  
                                      aten::convolution         0.31%     517.646us        20.32%      34.090ms     321.605us           106  
                                     aten::_convolution         1.53%       2.564ms        20.01%      33.572ms     316.721us           106  
autogr